# 35 · 需求、分级标注与评估

单元：S01,S02。先修：31–34。

来源：[王树森固定目录](https://github.com/wangshusen/SearchEngine/tree/7339c6b126e314abdb74a89adb033d39d8e3d37b)；[Manning等《Introduction to Information Retrieval》](https://nlp.stanford.edu/IR-book/)（补充原始教材；章节1、2、6、8）。这是 Lab 自编实验；王树森后四模块只有公开目录，不称原作者完整讲义。

证据边界：人工任务、会话与分级判断；不是 SciFact 分级标签。

学习方式：先预测每个对照，再只改一个变量，运行后解释差异。

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from reclab.learning.search_curriculum import *

## 任务先于指标
导航、信息与交易任务成功条件不同。预测：相同NDCG能否唯一确定满意度？下列会话是人工构造，停留和点击不直接等于成功。

In [2]:
sessions=pd.DataFrame([['navigation',1.,80,False],['navigation',1.,2,True],['research',.8,120,True]],columns=['task','ndcg','seconds','task_success'])
display(sessions)
assert sessions.iloc[0].ndcg==sessions.iloc[1].ndcg and sessions.iloc[0].task_success!=sessions.iloc[1].task_success

,task,ndcg,seconds,task_success
0,navigation,1.0,80,False
1,navigation,1.0,2,True
2,research,0.8,120,True


## 分档和手算
0无关、1部分、2相关、3直接满足；gain=2^rel−1，discount=log2(rank+1)。只交换前两条，预测分数变化。

In [3]:
judge={'a':3,'b':0,'c':1,'d':2}
orders=[['a','d','c'],['d','a','c']]
rows=[{'order':o,'ndcg':graded_ndcg(o,judge)} for o in orders]
manual=(3/np.log2(2)+7/np.log2(3)+1/np.log2(4))/(7+3/np.log2(3)+1/2)
assert np.isclose(rows[1]['ndcg'],manual)
display(pd.DataFrame(rows))

,order,ndcg
0,"[a, d, c]",1.000000
1,"[d, a, c]",0.842828


## 分歧和pooling
两个标注员对同一需求判断不一致，应记录指南、分歧和裁决。pool取两个系统top2的并集；pool外未判断不能默认为真实无关。预测加入新系统会漏掉哪些判断？

In [4]:
j1=np.array([3,0,1,2]); j2=np.array([2,0,2,2])
print('agreement',np.mean(j1==j2),'absolute disagreement',np.abs(j1-j2).mean())
pool=set(['a','d'])|set(['c','a']); unseen=set(DOCUMENTS)-pool
assert unseen=={'b'}
print('pool',sorted(pool),'unjudged',unseen)

agreement 0.5 absolute disagreement 0.5
pool ['a', 'c', 'd'] unjudged {'b'}


## 判断不完整的敏感性
只改变b的判断，其他保持不变。实际实验只能补标，不能为提高分数修改真值。SciFact旧读取二值化，25章继续作为真实契约课。

In [5]:
old=dict(judge); old['b']=0
new=dict(judge); new['b']=3
print('unjudged-as-zero',graded_ndcg(['b','a','d'],old),'counterfactual adjudication',graded_ndcg(['b','a','d'],new))

unjudged-as-zero 0.6298989693877386 counterfactual adjudication 1.0


## 本次保存结果的解读

保存结果中相同NDCG=1的两个导航会话成功与失败并存；交换前两条使分级NDCG从1降到约0.843。两位标注员完全一致率只有0.5；pool外b未判断，反事实补判断会显著改变分数。这些变化说明先定义需求与标注指南，再解释指标。

改参数后应以新输出重新判断，不把这段固定解读当作新的测量。

## 复盘与练习

写下输入、输出、保持不变的量与本章不能支持的结论。选一个上面的单变量参数先预测再重跑；保留失败例。回到搜索学习路径继续下一语义单元。